# Customer Churn Prediction — End-to-End ML Pipeline

**Author:** Senior ML Engineer | **Type:** Binary classification (tabular)

## Table of Contents
1. Problem Definition & Success Criteria
2. Setup & Configuration
3. Data Loading (Schema-validated)
4. Data Quality Audit
5. Exploratory Data Analysis (EDA)
6. Train / Validation / Test Split
7. Preprocessing Pipeline (leak-free)
8. Baseline & Model Comparison (Cross-Validation)
9. Hyperparameter Tuning
10. Final Evaluation on Held-out Test Set
11. Threshold Selection (business-driven)
12. Model Explainability
13. Model Persistence & Inference Function
14. Conclusions, Risks & Next Steps

> **Design principles used throughout:** (1) no data leakage — all fitting happens inside `sklearn.Pipeline` on training folds only, (2) reproducibility via a fixed seed and a single `CONFIG`, (3) metrics chosen to match the business problem, (4) test set touched **once**.

## 1. Problem Definition & Success Criteria

| Item | Definition |
|---|---|
| **Business problem** | A subscription company loses revenue when customers churn. We want to identify at-risk customers early so the retention team can contact them. |
| **Target** | `churn` (1 = customer left within the next 30 days, 0 = stayed) |
| **ML task** | Binary classification, imbalanced (~20–25% positives) |
| **Primary metric** | **PR-AUC** (average precision) — robust under class imbalance |
| **Secondary metrics** | ROC-AUC, F1, Recall @ chosen threshold, Brier score (calibration) |
| **Business constraint** | Retention team can contact only a limited number of customers, so *precision* matters; missing churners costs more than a wasted call, so *recall* matters too. We handle this explicitly via cost-based threshold selection (Section 11). |
| **Success criterion** | PR-AUC clearly above the naive baseline (= prevalence) and at least on par with a logistic regression baseline, with better-calibrated probabilities. |

**Note on data:** To keep this notebook fully self-contained and runnable anywhere, we generate a realistic synthetic dataset with mixed types, missing values and non-linear effects. To use real data, replace only the function `load_data()` in Section 3 — everything downstream stays unchanged.

## 2. Setup & Configuration

In [ ]:
import warnings
import json
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, brier_score_loss, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
@dataclass(frozen=True)
class Config:
    """Single source of truth for every tunable in this notebook."""
    seed: int = 42
    n_rows: int = 8000
    target: str = "churn"
    id_col: str = "customer_id"
    test_size: float = 0.20
    cv_folds: int = 5
    n_search_iter: int = 15
    primary_metric: str = "average_precision"   # sklearn scorer name (PR-AUC)
    # Business costs used for threshold selection (Section 11)
    cost_false_negative: float = 100.0   # lost customer lifetime value (simplified)
    cost_false_positive: float = 10.0    # cost of a retention offer / call
    artifacts_dir: Path = field(default_factory=lambda: Path("artifacts"))

CFG = Config()
CFG.artifacts_dir.mkdir(exist_ok=True)
RNG = np.random.default_rng(CFG.seed)
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

## 3. Data Loading (Schema-validated)

`load_data()` is the **only** function that knows where data comes from. It returns a DataFrame and we immediately validate it against an explicit **schema** (column names, types, allowed ranges). Failing fast here prevents silent corruption later.

In [ ]:
def load_data(cfg: Config) -> pd.DataFrame:
    """Generate a synthetic but realistic churn dataset.
    Replace the body with e.g. pd.read_csv(...) or a SQL query for real data."""
    rng = np.random.default_rng(cfg.seed)
    n = cfg.n_rows

    df = pd.DataFrame({
        cfg.id_col: np.arange(100000, 100000 + n),
        "tenure_months": rng.integers(1, 73, n),
        "monthly_charges": np.round(rng.normal(65, 25, n).clip(15, 150), 2),
        "support_tickets_90d": rng.poisson(1.2, n),
        "avg_weekly_logins": np.round(rng.gamma(3.0, 2.0, n), 2),
        "contract_type": rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20]),
        "payment_method": rng.choice(["card", "bank_transfer", "e-wallet", "cash"], n, p=[0.4, 0.25, 0.25, 0.1]),
        "plan": rng.choice(["basic", "standard", "premium"], n, p=[0.4, 0.4, 0.2]),
        "has_autopay": rng.choice([0, 1], n, p=[0.45, 0.55]),
    })

    # Ground-truth churn logit with non-linear effects & interactions
    logit = (
        -0.1
        - 0.035 * df["tenure_months"]
        + 0.012 * (df["monthly_charges"] - 65)
        + 0.30 * df["support_tickets_90d"]
        - 0.12 * df["avg_weekly_logins"]
        + np.where(df["contract_type"] == "month-to-month", 1.0, 0)
        - np.where(df["contract_type"] == "two-year", 0.8, 0)
        - 0.4 * df["has_autopay"]
        + np.where((df["plan"] == "premium") & (df["support_tickets_90d"] > 2), 0.9, 0)
        + rng.normal(0, 0.3, n)
    )
    p = 1 / (1 + np.exp(-logit))
    df[cfg.target] = (rng.random(n) < p).astype(int)

    # Inject realistic data problems: missing values + a few duplicates
    for col, frac in [("avg_weekly_logins", 0.06), ("payment_method", 0.03), ("monthly_charges", 0.02)]:
        df.loc[rng.random(n) < frac, col] = np.nan
    df = pd.concat([df, df.sample(25, random_state=cfg.seed)], ignore_index=True)  # duplicate rows
    return df

SCHEMA = {
    "customer_id":         dict(dtype="int",   min=0),
    "tenure_months":       dict(dtype="int",   min=0, max=120),
    "monthly_charges":     dict(dtype="float", min=0, max=1000),
    "support_tickets_90d": dict(dtype="int",   min=0, max=500),
    "avg_weekly_logins":   dict(dtype="float", min=0, max=500),
    "contract_type":       dict(dtype="cat",   allowed={"month-to-month", "one-year", "two-year"}),
    "payment_method":      dict(dtype="cat",   allowed={"card", "bank_transfer", "e-wallet", "cash"}),
    "plan":                dict(dtype="cat",   allowed={"basic", "standard", "premium"}),
    "has_autopay":         dict(dtype="int",   min=0, max=1),
    "churn":               dict(dtype="int",   min=0, max=1),
}

def validate_schema(df: pd.DataFrame, schema: dict) -> None:
    """Raise ValueError with a clear message if the data violates the contract."""
    errors = []
    missing_cols = set(schema) - set(df.columns)
    if missing_cols:
        errors.append(f"Missing columns: {sorted(missing_cols)}")
    for col, rule in schema.items():
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if rule["dtype"] == "cat":
            bad = set(s.unique()) - rule["allowed"]
            if bad:
                errors.append(f"{col}: unexpected categories {bad}")
        else:
            if "min" in rule and (s < rule["min"]).any():
                errors.append(f"{col}: values below {rule['min']}")
            if "max" in rule and (s > rule["max"]).any():
                errors.append(f"{col}: values above {rule['max']}")
    if errors:
        raise ValueError("Schema validation failed:\n - " + "\n - ".join(errors))
    print("Schema validation passed.")

raw = load_data(CFG)
validate_schema(raw, SCHEMA)
print(f"Shape: {raw.shape}")
raw.head()

## 4. Data Quality Audit

Before any modelling we quantify: duplicates, missingness, cardinality and basic statistics. Decisions made here are applied **only to the training data logic** later (imputation is fitted inside the pipeline), while deterministic cleaning (dropping exact duplicates) is safe to do now.

In [ ]:
def audit(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    return out

print(f"Exact duplicate rows : {raw.duplicated().sum()}")
print(f"Duplicate customer_id: {raw[CFG.id_col].duplicated().sum()}")
audit(raw)

In [ ]:
# Deterministic cleaning (no statistics learned from data -> no leakage risk)
df = raw.drop_duplicates().reset_index(drop=True)
assert df[CFG.id_col].is_unique, "customer_id must be unique after de-duplication"
print(f"After de-duplication: {df.shape}")

prevalence = df[CFG.target].mean()
print(f"Churn prevalence (naive PR-AUC baseline): {prevalence:.3f}")
df.describe().T

## 5. Exploratory Data Analysis (EDA)
Goal: understand signal, spot non-linearity, and justify modelling choices — not to produce pretty plots.

In [ ]:
NUM_COLS = ["tenure_months", "monthly_charges", "support_tickets_90d", "avg_weekly_logins"]
CAT_COLS = ["contract_type", "payment_method", "plan"]
BIN_COLS = ["has_autopay"]
FEATURES = NUM_COLS + CAT_COLS + BIN_COLS

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), NUM_COLS):
    for label, grp in df.groupby(CFG.target):
        ax.hist(grp[col].dropna(), bins=30, alpha=0.55, density=True, label=f"churn={label}")
    ax.set_title(col); ax.legend()
plt.suptitle("Numeric feature distributions by target", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, col in zip(axes, CAT_COLS + BIN_COLS):
    rate = df.groupby(col, dropna=False)[CFG.target].mean().sort_values()
    rate.plot.barh(ax=ax, color="#4C72B0")
    ax.axvline(prevalence, color="red", ls="--", lw=1)
    ax.set_title(f"Churn rate by {col}")
plt.tight_layout(); plt.show()

# Is missingness informative? (a common hidden signal)
for col in ["avg_weekly_logins", "payment_method", "monthly_charges"]:
    rate = df.groupby(df[col].isna())[CFG.target].mean()
    print(f"Churn rate | {col} missing={rate.get(True, np.nan):.3f}  vs present={rate.get(False, np.nan):.3f}")

In [ ]:
corr = df[NUM_COLS + BIN_COLS + [CFG.target]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); ax.set_title("Spearman correlation"); ax.grid(False)
plt.tight_layout(); plt.show()

**EDA takeaways (what we act on):**
- Shorter tenure, month-to-month contracts, more support tickets and lower engagement all raise churn risk → strong, mostly monotonic signal.
- Missing values exist in 3 columns → impute inside the pipeline and add **missing-indicator** flags.
- Relationships look non-linear / interactive → compare a linear baseline with tree-based models.
- Class imbalance is moderate → use stratified splits and PR-AUC; class weighting is tested, not assumed.

## 6. Train / Validation / Test Split

- **Stratified** to preserve churn rate.
- The **test set is locked away** until Section 10. All model selection uses cross-validation on the training portion only (so the CV folds act as validation sets).

In [ ]:
X = df[FEATURES]
y = df[CFG.target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CFG.test_size, stratify=y, random_state=CFG.seed
)
print(f"Train: {X_train.shape}, churn rate {y_train.mean():.3f}")
print(f"Test : {X_test.shape}, churn rate {y_test.mean():.3f}")
CV = StratifiedKFold(n_splits=CFG.cv_folds, shuffle=True, random_state=CFG.seed)

## 7. Preprocessing Pipeline (leak-free)

Everything that *learns from data* (imputation medians, scaling stats, category lists) lives inside a `ColumnTransformer`, which is itself the first step of a `Pipeline`. During cross-validation sklearn refits it on each training fold only — **this is what prevents leakage**.

A small custom transformer demonstrates how to add domain feature engineering in a pipeline-safe way.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Stateless, pipeline-safe domain features. Works on DataFrames."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["charges_x_tenure"] = X["monthly_charges"] * X["tenure_months"]            # lifetime spend proxy
        X["tickets_per_year_tenure"] = X["support_tickets_90d"] / (X["tenure_months"] / 12 + 1)
        X["is_new_customer"] = (X["tenure_months"] <= 6).astype(int)
        return X

ENG_NUM = NUM_COLS + ["charges_x_tenure", "tickets_per_year_tenure"]
ENG_BIN = BIN_COLS + ["is_new_customer"]

def make_preprocessor(scale: bool) -> ColumnTransformer:
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), ENG_NUM),
            ("cat", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), CAT_COLS),
            ("bin", "passthrough", ENG_BIN),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

def make_pipeline(model, scale: bool) -> Pipeline:
    return Pipeline([
        ("features", FeatureEngineer()),
        ("preprocess", make_preprocessor(scale)),
        ("model", model),
    ])

# Sanity check: what does the model actually see?
_demo = make_pipeline(LogisticRegression(), scale=True).fit(X_train, y_train)
print("Transformed feature count:", _demo[:-1].transform(X_train).shape[1])
_demo[:-1].transform(X_train.head(3))

## 8. Baseline & Model Comparison (Cross-Validation)

We always start with **dumb baselines** so every later gain is measurable:
1. `DummyClassifier` (predicts prevalence) — the floor.
2. Logistic Regression — strong interpretable baseline.
3. Random Forest, 4. Gradient Boosting — non-linear candidates.

All evaluated with identical stratified CV and multiple metrics.

In [ ]:
CANDIDATES = {
    "Dummy (prior)": make_pipeline(DummyClassifier(strategy="prior"), scale=False),
    "LogReg": make_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced"), scale=True),
    "RandomForest": make_pipeline(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                               n_jobs=-1, random_state=CFG.seed), scale=False),
    "HistGradBoost": make_pipeline(
        HistGradientBoostingClassifier(learning_rate=0.06, max_iter=300, early_stopping=True,
                                       random_state=CFG.seed), scale=False),
}
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "f1": "f1", "neg_brier": "neg_brier_score"}

rows = []
for name, pipe in CANDIDATES.items():
    res = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
    rows.append({
        "model": name,
        "PR-AUC": res["test_pr_auc"].mean(), "PR-AUC std": res["test_pr_auc"].std(),
        "ROC-AUC": res["test_roc_auc"].mean(), "F1@0.5": res["test_f1"].mean(),
        "Brier": -res["test_neg_brier"].mean(), "fit_s": res["fit_time"].mean(),
    })
cv_table = pd.DataFrame(rows).set_index("model").sort_values("PR-AUC", ascending=False).round(4)
cv_table

In [ ]:
ax = cv_table["PR-AUC"].sort_values().plot.barh(xerr=cv_table.loc[cv_table["PR-AUC"].sort_values().index, "PR-AUC std"],
                                                 color="#55A868", figsize=(7, 3.5))
ax.axvline(prevalence, color="red", ls="--", label=f"prevalence = {prevalence:.2f}")
ax.set_xlabel("CV PR-AUC (mean ± std)"); ax.set_title("Model comparison"); ax.legend()
plt.tight_layout(); plt.show()

## 9. Hyperparameter Tuning

Logistic regression and the tree models score within noise of each other on PR-AUC (compare the ± std above), but HistGradientBoosting has the best Brier score (calibration) and can capture interactions that a linear model cannot, so we tune it. If the gap were larger in favour of LogReg we would tune that instead. We tune with `RandomizedSearchCV` — **on the training set only**, same CV scheme, optimising the primary metric. Randomised search is more sample-efficient than grid search for the same budget.

In [ ]:
param_dist = {
    "model__learning_rate":     [0.02, 0.04, 0.06, 0.1],
    "model__max_iter":          [150, 300, 500],
    "model__max_depth":         [None, 3, 5, 8],
    "model__min_samples_leaf":  [10, 20, 40, 80],
    "model__l2_regularization": [0.0, 0.1, 1.0, 5.0],
    "model__max_leaf_nodes":    [15, 31, 63],
}
search = RandomizedSearchCV(
    estimator=make_pipeline(HistGradientBoostingClassifier(early_stopping=True, random_state=CFG.seed), scale=False),
    param_distributions=param_dist,
    n_iter=CFG.n_search_iter, scoring=CFG.primary_metric, cv=CV,
    n_jobs=-1, random_state=CFG.seed, refit=True,
)
search.fit(X_train, y_train)
print(f"Best CV PR-AUC: {search.best_score_:.4f}")
print("Best params:", json.dumps(search.best_params_, indent=2))
best_model = search.best_estimator_

## 10. Final Evaluation on Held-out Test Set

The test set is used **once**, with the tuned model. We report discrimination (ROC/PR), calibration and confusion matrix, plus a **bootstrap confidence interval** because a single point estimate hides uncertainty.

In [ ]:
proba_test = best_model.predict_proba(X_test)[:, 1]

def bootstrap_ci(y_true, y_score, metric, n=500, seed=CFG.seed):
    rng = np.random.default_rng(seed)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].min() == y_true[idx].max():
            continue
        vals.append(metric(y_true[idx], y_score[idx]))
    return np.percentile(vals, [2.5, 97.5])

pr, roc = average_precision_score(y_test, proba_test), roc_auc_score(y_test, proba_test)
print(f"Test PR-AUC : {pr:.4f}  95% CI {bootstrap_ci(y_test, proba_test, average_precision_score).round(4)}  (baseline {y_test.mean():.3f})")
print(f"Test ROC-AUC: {roc:.4f}  95% CI {bootstrap_ci(y_test, proba_test, roc_auc_score).round(4)}")
print(f"Test Brier  : {brier_score_loss(y_test, proba_test):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=axes[0], name="Tuned HGB")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=axes[1], name="Tuned HGB")
axes[1].axhline(y_test.mean(), color="red", ls="--", label="baseline")
axes[1].legend()
CalibrationDisplay.from_predictions(y_test, proba_test, n_bins=10, ax=axes[2], name="Tuned HGB")
axes[0].set_title("ROC"); axes[1].set_title("Precision-Recall"); axes[2].set_title("Calibration")
plt.tight_layout(); plt.show()

## 11. Threshold Selection (business-driven)

A default threshold of 0.5 is rarely optimal. We pick the threshold that **minimises expected cost**:

`cost = FN × cost_false_negative + FP × cost_false_positive`

To avoid optimistic bias, the threshold is chosen using **out-of-fold predictions on the training set** and then applied once to the test set.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

oof_proba = cross_val_predict(clone(best_model), X_train, y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]

def expected_cost(y_true, y_pred, c_fn=CFG.cost_false_negative, c_fp=CFG.cost_false_positive):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn * c_fn + fp * c_fp

grid = np.linspace(0.05, 0.95, 91)
costs = np.array([expected_cost(y_train, (oof_proba >= t).astype(int)) for t in grid])
BEST_THRESHOLD = float(grid[costs.argmin()])

plt.plot(grid, costs); plt.axvline(BEST_THRESHOLD, color="red", ls="--", label=f"best = {BEST_THRESHOLD:.2f}")
plt.axvline(0.5, color="grey", ls=":", label="default 0.5")
plt.xlabel("Threshold"); plt.ylabel("Expected cost (train OOF)"); plt.legend(); plt.title("Cost vs threshold")
plt.tight_layout(); plt.show()

pred_test = (proba_test >= BEST_THRESHOLD).astype(int)
default_cost = expected_cost(y_test, (proba_test >= 0.5).astype(int))
tuned_cost = expected_cost(y_test, pred_test)
print(f"Test cost @0.5 : {default_cost:,.0f}")
print(f"Test cost @{BEST_THRESHOLD:.2f}: {tuned_cost:,.0f}  ({(1 - tuned_cost / default_cost) * 100:.1f}% saving)")
print(f"Precision {precision_score(y_test, pred_test):.3f} | Recall {recall_score(y_test, pred_test):.3f} | F1 {f1_score(y_test, pred_test):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["stay", "churn"], cmap="Blues")
plt.grid(False); plt.title("Confusion matrix (test)"); plt.show()
print(classification_report(y_test, pred_test, target_names=["stay", "churn"]))

## 12. Model Explainability

**Permutation importance** on the test set, computed on the *raw input columns* (the whole pipeline is permuted, so feature engineering and encoding are accounted for). It is model-agnostic and measures real predictive contribution, unlike impurity-based importance which is biased toward high-cardinality features.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring="average_precision",
                              n_repeats=10, random_state=CFG.seed, n_jobs=-1)
imp = (pd.DataFrame({"feature": X_test.columns, "mean": perm.importances_mean, "std": perm.importances_std})
         .sort_values("mean"))
plt.barh(imp["feature"], imp["mean"], xerr=imp["std"], color="#C44E52")
plt.xlabel("Drop in PR-AUC when shuffled"); plt.title("Permutation importance (test)")
plt.tight_layout(); plt.show()
imp.sort_values("mean", ascending=False).reset_index(drop=True)

## 13. Model Persistence & Inference Function

We save the **entire pipeline** (feature engineering + preprocessing + model) together with metadata, so production inference can never drift from training logic. The `predict_churn` function also validates its input.

In [ ]:
import sys, sklearn, datetime as dt

MODEL_PATH = CFG.artifacts_dir / "churn_pipeline.joblib"
META_PATH = CFG.artifacts_dir / "model_card.json"

joblib.dump(best_model, MODEL_PATH)
model_card = {
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "sklearn": sklearn.__version__,
    "features": FEATURES,
    "threshold": BEST_THRESHOLD,
    "best_params": {k: (v if v is None or isinstance(v, (int, float, str)) else str(v)) for k, v in search.best_params_.items()},
    "cv_pr_auc": round(float(search.best_score_), 4),
    "test_pr_auc": round(float(pr), 4),
    "test_roc_auc": round(float(roc), 4),
    "train_rows": int(len(X_train)),
    "seed": CFG.seed,
}
META_PATH.write_text(json.dumps(model_card, indent=2))
print(json.dumps(model_card, indent=2))

In [ ]:
def predict_churn(records: pd.DataFrame, model_path=MODEL_PATH, meta_path=META_PATH) -> pd.DataFrame:
    """Score new customers. Input must contain the training feature columns."""
    model = joblib.load(model_path)
    meta = json.loads(Path(meta_path).read_text())
    missing = set(meta["features"]) - set(records.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    proba = model.predict_proba(records[meta["features"]])[:, 1]
    out = records.copy()
    out["churn_probability"] = proba.round(4)
    out["churn_flag"] = (proba >= meta["threshold"]).astype(int)
    return out

# Round-trip test: reloaded model must reproduce in-memory predictions exactly
reloaded = predict_churn(X_test.head(200))
assert np.allclose(reloaded["churn_probability"], best_model.predict_proba(X_test.head(200))[:, 1].round(4)), "Persistence mismatch!"
print("Round-trip check passed.")
reloaded[["tenure_months", "contract_type", "support_tickets_90d", "churn_probability", "churn_flag"]].head()

## 14. Conclusions, Risks & Next Steps

**Summary**
- All real models far exceed the prevalence baseline. LogReg, RandomForest and gradient boosting are statistically close on PR-AUC (Section 8); gradient boosting gives the best-calibrated probabilities, which matters for cost-based thresholding. Treat the model choice as a judgement call, not a decisive win.
- A cost-based threshold reduces expected business cost versus the default 0.5 cut-off.
- The pipeline is leak-free, reproducible (single seed/config) and persisted as one artifact with a model card.

**Known risks / limitations**
- Data here is synthetic; real data will bring label noise, drift and leakage traps (e.g. features recorded *after* the churn event). Audit feature timestamps before deployment.
- Costs in `Config` are illustrative — replace with finance-approved values.
- Random split ignores time. If real data is temporal, use a **time-based split** (train on past, test on future).

**Production next steps**
1. Replace `load_data()` with the real source; re-run the schema validation.
2. Add **drift monitoring** (PSI on key features, score distribution) and scheduled re-calibration.
3. Add fairness checks across customer segments, if policy requires.
4. Wrap `predict_churn` in an API/batch job; log inputs and predictions for auditing.
5. Run an A/B test of the retention campaign to measure *causal* uplift, not just predictive accuracy.